# EduInsight — Notebook 4: Model Development

## Objective

Compare an interpretable Logistic Regression baseline with a Random Forest,
select a candidate using cross-validation on the training partition, and
evaluate the selected model once on an untouched holdout set.

In [1]:
from pathlib import Path
import sys


def locate_project_root(
    start: Path | None = None,
) -> Path:
    current = (
        start or Path.cwd()
    ).resolve()

    for candidate in (
        current,
        *current.parents,
    ):
        if (
            (candidate / "src").is_dir()
            and (
                candidate / "notebooks"
            ).is_dir()
        ):
            return candidate

    raise FileNotFoundError(
        "Could not locate the project root. Open VS Code from the repository "
        "folder and run the notebook again."
    )


PROJECT_ROOT = locate_project_root()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(PROJECT_ROOT),
    )


RAW_DATA_DIR = (
    PROJECT_ROOT
    / "data"
    / "raw"
)

PROCESSED_DATA_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)

MODEL_DIR = (
    PROJECT_ROOT
    / "models"
)


SELECTED_MODULE = "BBB"
SELECTED_PRESENTATION = "2013J"
CUTOFF_DAY = 30


print(
    f"Project root: "
    f"{PROJECT_ROOT}"
)

Project root: D:\Work\GIT\eduinsight-student-success


## 1. Imports and processed-data loading

In [2]:
import pandas as pd

from src.model_training import (
    MODEL_FEATURES,
    build_candidate_models,
    build_feature_summary,
    create_holdout_split,
    cross_validate_candidates,
    evaluate_classifier,
    save_model_artifact,
)


processed_path = (
    PROCESSED_DATA_DIR
    / "student_features.csv"
)


if not processed_path.is_file():
    raise FileNotFoundError(
        "Processed data is missing. "
        "Run 02_data_preparation.ipynb first."
    )


data = pd.read_csv(
    processed_path
)


print(
    f"Modelling rows: "
    f"{len(data):,}"
)

Modelling rows: 1,825


## 2. Feature policy

The first model intentionally excludes gender, region, age band, disability,
and deprivation-band variables. The selected features describe early
behaviour, study history, and registration timing.

In [3]:
feature_table = pd.DataFrame(
    {
        "model_feature": (
            MODEL_FEATURES
        )
    }
)


feature_table

,model_feature
0,num_of_prev_attempts
1,studied_credits
2,date_registration
3,total_clicks_30
4,active_days_30
5,unique_resources_30
6,avg_clicks_per_active_day_30
7,assessments_submitted_30
8,avg_score_30
9,late_submissions_30


## 3. Reserve an untouched holdout set

In [4]:
(
    X_train,
    X_test,
    y_train,
    y_test,
) = create_holdout_split(
    data,
    test_size=0.20,
    random_state=42,
)


print(
    f"Training rows: "
    f"{len(X_train):,}"
)

print(
    f"Holdout rows: "
    f"{len(X_test):,}"
)

print(
    "Training target distribution:"
)

print(
    y_train
    .value_counts(
        normalize=True
    )
    .sort_index()
    .round(3)
)

Training rows: 1,460
Holdout rows: 365
Training target distribution:
support_needed
0    0.587
1    0.413
Name: proportion, dtype: float64


The holdout set is created before model comparison. It is not used to choose
the model, which reduces optimistic evaluation bias.

## 4. Compare candidate models with stratified cross-validation

In [5]:
candidate_models = (
    build_candidate_models(
        random_state=42
    )
)


cv_results = (
    cross_validate_candidates(
        candidate_models,
        X_train,
        y_train,
        n_splits=5,
        random_state=42,
    )
)


cv_results.round(3)

,model,mean_cv_f1_support,mean_cv_roc_auc,mean_cv_average_precision,mean_cv_balanced_accuracy
0,Random Forest,0.635,0.740,0.696,0.691
1,Logistic Regression,0.629,0.748,0.702,0.686


## 5. Select and fit the best cross-validated candidate

In [6]:
selected_model_name = str(
    cv_results.iloc[0][
        "model"
    ]
)


selected_model = (
    candidate_models[
        selected_model_name
    ]
)


selected_model.fit(
    X_train,
    y_train,
)


print(
    f"Selected model: "
    f"{selected_model_name}"
)

Selected model: Random Forest


## 6. Evaluate once on the untouched holdout set

In [7]:
holdout_metrics = (
    evaluate_classifier(
        selected_model,
        X_test,
        y_test,
        threshold=0.50,
    )
)


metric_names = [
    "balanced_accuracy",
    "precision_support",
    "recall_support",
    "f1_support",
    "roc_auc",
    "average_precision",
]


pd.Series(
    {
        name: holdout_metrics[
            name
        ]
        for name in metric_names
    },
    name="holdout_score",
).round(3).to_frame()

,holdout_score
balanced_accuracy,0.666
precision_support,0.611
recall_support,0.603
f1_support,0.607
roc_auc,0.740
average_precision,0.678


In [8]:
confusion = pd.DataFrame(
    holdout_metrics[
        "confusion_matrix"
    ],
    index=[
        "Actual no flag",
        "Actual support flag"
    ],
    columns=[
        "Predicted no flag",
        "Predicted support flag"
    ]
)

confusion

,Predicted no flag,Predicted support flag
Actual no flag,156,58
Actual support flag,60,91


In [9]:
classification_report_table = (
    pd.DataFrame(
        holdout_metrics[
            "classification_report"
        ]
    )
    .transpose()
    .round(3)
)

classification_report_table

,precision,recall,f1-score,support
No support flag,0.722,0.729,0.726,214.000
Support flag,0.611,0.603,0.607,151.000
accuracy,0.677,0.677,0.677,0.677
macro avg,0.666,0.666,0.666,365.000
weighted avg,0.676,0.677,0.676,365.000


## 7. Save the complete model artifact

The saved artifcat contains the fitted preprocessing-and-model pipeline, feature order, model metadata, holdout metrics, and numeric defaults used by the Streamlit form.

In [10]:
MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

model_path = (
    MODEL_DIR
    / "early_support_model.joblib"
)

metadata = {
    "model_name": (
        selected_model_name
    ),
    "module": (
        SELECTED_MODULE
    ),
    "presentation": (
        SELECTED_PRESENTATION
    ),
    "cutoff_day": (
        CUTOFF_DAY
    ),
    "target_definition": (
        "1 = Fail or Withdrawn after "
        "the day-30 eligibility filter"
    ),
    "random_state": 42,
}


save_model_artifact(
    selected_model,
    model_path,
    feature_columns=(
        MODEL_FEATURES
    ),
    metadata=metadata,
    metrics=holdout_metrics,
    feature_summary=(
        build_feature_summary(
            X_train
        )
    ),
)


print(
    f"Saved model artifact: "
    f"{model_path}"
)

Saved model artifact: D:\Work\GIT\eduinsight-student-success\models\early_support_model.joblib


## 8. Professional interpretation checklist

Before adding results to the README:

- report the holdout metrics without exaggeration;
- explain false positives and false negatives;
- state that the result is not causal;
- state that the model is trained on one historical presentation;
- avoid describing the support score as a guaranteed probability of failure.